# Latent decision boundary — baseline lab

Binary classification on fourteen continuous features (`f0`–`f13`). The TASK stresses a **latent nonlinear** decision surface and **label noise** (~8%): marginal correlations with `y` stay weak, so plain linear models on raw features look nearly random.

**Deliverable:** `submission_pred_baseline.csv` with column `pred_proba`, same row order as `test_public.csv`, from a model refit on **train ∪ val** after choosing hyperparameters on **val** only.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*ConvergenceWarning.*")

## Paths and load

In [2]:
ROOT = Path(".") if Path("train.csv").exists() else Path("demos/comparison_latent")
FEAT = [f"f{i}" for i in range(14)]

train = pd.read_csv(ROOT / "train.csv")
val = pd.read_csv(ROOT / "val.csv")
test = pd.read_csv(ROOT / "test_public.csv")
len(train), len(val), len(test)

(3200, 800, 900)

## Class balance · marginal correlations

If every $|\mathrm{corr}(f_k,y)|$ is small, a single linear score on $x$ underfits; we need explicit interactions/curvature or a nonlinear learner.

In [3]:
print("Train positivity rate:", train["y"].mean())
print("Val   positivity rate:", val["y"].mean())

cors = {
    c: float(np.corrcoef(train[c].to_numpy(), train["y"].to_numpy(), rowvar=True)[0, 1])
    for c in FEAT
}
sorted(cors.items(), key=lambda kv: abs(kv[1]), reverse=True)

Train positivity rate: 0.5109375
Val   positivity rate: 0.5075


[('f6', -0.08531041133262553),
 ('f10', 0.06803893624951149),
 ('f8', 0.04476345036944852),
 ('f9', 0.03760131209215811),
 ('f5', 0.02014235686745461),
 ('f1', 0.018506092412505757),
 ('f13', -0.014464022177875008),
 ('f7', 0.013942219591899691),
 ('f2', -0.008818302193176369),
 ('f4', 0.006521576039103986),
 ('f0', -0.0059469786814706355),
 ('f11', 0.003941696864101156),
 ('f12', -0.0035543247698732777),
 ('f3', -0.0016378275832336063)]

## Model probes (fit on train, score on val)

- **Raw logistic** — sanity baseline.
- **Degree-2 polynomial expansion** (14 → 119 non-constant terms): captures squares and pairwise interactions without hand-engineering the generator.
- **Elastic net** on the expanded features encourages sparsity/stability under label noise. **`C=0.02`**, **`l1_ratio=0.7`** were chosen from a validation sweep favouring more regularisation than plain L2 on this expanded space (reduces overfit to flipped labels).

In [4]:
Xtr, ytr = train[FEAT].to_numpy(dtype=np.float64), train["y"].to_numpy()
Xva, yva = val[FEAT].to_numpy(dtype=np.float64), val["y"].to_numpy()

def report(name: str, pipe: Pipeline) -> float:
    pipe.fit(Xtr, ytr)
    p = pipe.predict_proba(Xva)[:, 1]
    auc = float(roc_auc_score(yva, p))
    print(f"{name:32s} val ROC-AUC = {auc:.4f}")
    return auc

raw_lr = Pipeline(
    [
        ("sc", StandardScaler()),
        ("m", LogisticRegression(max_iter=4000, C=1.0, random_state=0)),
    ]
)
report("Raw standardized logistic", raw_lr)

poly_l2 = Pipeline(
    [
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("sc", StandardScaler()),
        ("m", LogisticRegression(max_iter=12000, C=0.01, solver="lbfgs", random_state=0)),
    ]
)
report("Poly(2) + L2 logistic", poly_l2)

C_EN, L1_EN = 0.02, 0.7
poly_en = Pipeline(
    [
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("sc", StandardScaler()),
        (
            "m",
            LogisticRegression(
                max_iter=15000,
                C=C_EN,
                penalty="elasticnet",
                solver="saga",
                l1_ratio=L1_EN,
                random_state=0,
            ),
        ),
    ]
)
val_auc_chosen = report("Poly(2) + elastic net (chosen)", poly_en)

Raw standardized logistic        val ROC-AUC = 0.5739
Poly(2) + L2 logistic            val ROC-AUC = 0.6224
Poly(2) + elastic net (chosen)   val ROC-AUC = 0.6526


## Submission: refit on train ∪ val

Hyperparameters fixed from the step above; **no peeking** at the private test. Writes **`submission_pred_baseline.csv`** only (MCP companion artifacts are not used here).

In [5]:
combo = pd.concat([train, val], axis=0, ignore_index=True)
Xall, yall = combo[FEAT].to_numpy(dtype=np.float64), combo["y"].to_numpy()
Xte = test[FEAT].to_numpy(dtype=np.float64)

final = Pipeline(
    [
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("sc", StandardScaler()),
        (
            "m",
            LogisticRegression(
                max_iter=15000,
                C=C_EN,
                penalty="elasticnet",
                solver="saga",
                l1_ratio=L1_EN,
                random_state=0,
            ),
        ),
    ]
)
final.fit(Xall, yall)
pred = final.predict_proba(Xte)[:, 1]

out = pd.DataFrame({"pred_proba": pred})
out_path = ROOT / "submission_pred_baseline.csv"
out.to_csv(out_path, index=False)
len(out), out["pred_proba"].min(), out["pred_proba"].max(), str(out_path.resolve())

(900,
 np.float64(0.1811041814150988),
 np.float64(0.922175541244873),
 'C:\\Users\\goela\\Projects\\LLM_Jupyter\\demos\\comparison_latent\\submission_pred_baseline.csv')

## Summary (validation estimate only; test scored externally)

- **Estimated validation ROC-AUC** for the chosen **Poly(2) + elastic-net** pipeline (train fit, val evaluation): **0.6526** (see probe cell; variable `val_auc_chosen`).
- **Why it should generalise:** the expanded quadratic surface absorbs multiplicative/tanh-like structure without fixing a parametric form; elastic net trims unstable directions when labels are noisy. Refitting on train ∪ val uses all labelled data for the final $\Pr(y=1)$ **ranking**; calibration on a new cohort can still drift.
- **Failure modes:** if the true boundary needs beyond-quadratic smooth terms or heavy-tail artifacts, tree ensembles tuned on more data can win; we kept this baseline interpretable and fast under `nbconvert --execute`.